# Food Calorie Estimator — Data Preparation
## Dataset: Nutrition5k (Google Research)
This notebook covers:
- Loading the dataset
- Linking images to metadata
- Cleaning & validating images
- Resize + Normalization
- Data Augmentation
- Train / Validation / Test Split

## 1. Import Libraries

In [1]:
import requests
import os
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
from torchvision import transforms

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / 'data').exists() and (PROJECT_ROOT.parent / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data'
IMAGE_DIR = DATA_DIR / 'images'
DATA_DIR.mkdir(exist_ok=True)
IMAGE_DIR.mkdir(exist_ok=True)

print("Libraries imported")


✅ Libraries imported


## 2. Load Metadata (CSV Files)
We read both CSV files manually since each row has a variable number of columns (depending on the number of ingredients per dish).

In [2]:
dish_data = []

for filename in ['nutrition5k_dataset_metadata_dish_metadata_cafe1.csv',
                 'nutrition5k_dataset_metadata_dish_metadata_cafe2.csv']:
    with open(DATA_DIR / filename, 'r') as f:
        for line in f:
            parts = line.strip().split(',')
            dish_id = parts[0]
            calories = parts[1]
            dish_data.append((dish_id, calories))

print(f"Total dishes found: {len(dish_data)}")
print("Sample:", dish_data[:3])

Total dishes found: 5006
Sample: [('dish_1561662216', '300.794281'), ('dish_1562688426', '137.569992'), ('dish_1561662054', '419.438782')]


## 3. Download Images
We download only the overhead RGB image (`rgb.png`) for each dish directly from Google Cloud Storage.
We limit to **1500 dishes** to keep the dataset manageable (~1-2 GB).

In [3]:
IMAGE_DIR.mkdir(exist_ok=True)

base_url = "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead"

for i, (dish_id, calories) in enumerate(dish_data[:1500]):
    save_path = IMAGE_DIR / f"{dish_id}.png"
    if os.path.exists(save_path):
        continue

    try:
        response = requests.get(f"{base_url}/{dish_id}/rgb.png", timeout=10)
        if response.status_code == 200:
            with open(save_path, 'wb') as f:
                f.write(response.content)
            print(f"✅ {i+1}/1500 - {dish_id}")
        else:
            print(f"❌ {dish_id}")
    except:
        print(f"⏭️ skipped (timeout) - {dish_id}")
        continue

print(f"\nTotal images downloaded: {len(list(IMAGE_DIR.glob('*.png')))}")

❌ dish_1562688426
❌ dish_1563379132
❌ dish_1550795690
❌ dish_1550876012
❌ dish_1551565034
❌ dish_1550860747
❌ dish_1566245398
❌ dish_1563381680
❌ dish_1550778583
❌ dish_1568144828
❌ dish_1550708440
❌ dish_1551307090
❌ dish_1551226363
❌ dish_1551374440
❌ dish_1551492561
❌ dish_1551237016
❌ dish_1551489812
❌ dish_1551494178
❌ dish_1551233434
❌ dish_1551390304
❌ dish_1568144803
❌ dish_1551487158
❌ dish_1561997365
❌ dish_1551395233
❌ dish_1550775393
❌ dish_1551376970
❌ dish_1564082178
❌ dish_1550707756
❌ dish_1551227216
❌ dish_1551492702
❌ dish_1563378110
❌ dish_1550771267
❌ dish_1551390215
❌ dish_1551565884
❌ dish_1564415022
❌ dish_1551391034
❌ dish_1551391302
❌ dish_1551222877
❌ dish_1551223114
❌ dish_1566242164
❌ dish_1550775363
❌ dish_1566502211
❌ dish_1551314967
❌ dish_1551125357
❌ dish_1551389684
❌ dish_1551563213
❌ dish_1550775318
❌ dish_1550863247
❌ dish_1550772027
❌ dish_1550797833
❌ dish_1551222759
❌ dish_1551380950
❌ dish_1551306452
❌ dish_1551390979
❌ dish_1550707156
❌ dish_155

## 4. Link Images to Metadata
We match each downloaded image to its corresponding calorie value using the `dish_id`.

In [4]:
dish_data_dict = {dish_id: float(calories) for dish_id, calories in dish_data}

dataset = []
for img_path in IMAGE_DIR.glob('*.png'):
    img_file = img_path.name
    dish_id = img_file.replace('.png', '')
    if dish_id in dish_data_dict:
        dataset.append({
            'dish_id': dish_id,
            'image_path': str(img_path),
            'calories': dish_data_dict[dish_id]
        })

df = pd.DataFrame(dataset)
print(f"Dataset size: {len(df)}")
print(df.head())

Dataset size: 1021
           dish_id                  image_path    calories
0  dish_1556575558  images/dish_1556575558.png  103.300003
1  dish_1557853154  images/dish_1557853154.png   18.810001
2  dish_1557853229  images/dish_1557853229.png  195.929993
3  dish_1557861216  images/dish_1557861216.png    0.000000
4  dish_1557861697  images/dish_1557861697.png   13.144000


## 5. Data Cleaning
- Remove dishes with **0 calories** (invalid entries)
- Validate that all images can be opened correctly

In [5]:
# Remove zero-calorie entries
df = df[df['calories'] > 0].reset_index(drop=True)
print(f"After removing zero calories: {len(df)} images")

# Validate images
def is_valid_image(path):
    try:
        img = Image.open(path)
        img.verify()
        return True
    except:
        return False

df = df[df['image_path'].apply(is_valid_image)].reset_index(drop=True)
print(f"After image validation: {len(df)} images")

print(f"\nCalories stats:")
print(df['calories'].describe())

After removing zero calories: 1020 images
After image validation: 1020 images

Calories stats:
count    1020.000000
mean      260.431425
std       217.427796
min         1.150000
25%        75.497498
50%       210.950302
75%       385.351120
max      1324.084961
Name: calories, dtype: float64


## 6. Resize & Normalization
- Resize all images to **224x224** (standard input size for CNNs)
- Normalize pixel values to **[0, 1]**

In [6]:
def preprocess_image(path):
    img = Image.open(path).convert('RGB')
    img = img.resize((224, 224))
    img_array = np.array(img) / 255.0  # normalize to [0, 1]
    return img_array

# Test on one image
sample = preprocess_image(df['image_path'][0])
print(f"Image shape: {sample.shape}")
print(f"Min value: {sample.min():.2f} | Max value: {sample.max():.2f}")

Image shape: (224, 224, 3)
Min value: 0.00 | Max value: 1.00


## 7. Data Augmentation
To improve model generalization, we define augmentation transforms:
- Random Horizontal Flip
- Random Rotation (±15°)
- Color Jitter (brightness adjustment)

In [7]:
augmentation = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2),
])

print("✅ Augmentation pipeline defined")
print(augmentation)

✅ Augmentation pipeline defined
Compose(
    RandomHorizontalFlip(p=0.5)
    RandomRotation(degrees=[-15.0, 15.0], interpolation=nearest, expand=False, fill=0)
    ColorJitter(brightness=(0.8, 1.2), contrast=None, saturation=None, hue=None)
)


## 8. Train / Validation / Test Split
We split the dataset into:
- **70%** Training
- **15%** Validation
- **15%** Testing

In [8]:
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42)
val_df, test_df   = train_test_split(temp_df, test_size=0.50, random_state=42)

print(f"Train : {len(train_df)} images ({len(train_df)/len(df)*100:.1f}%)")
print(f"Val   : {len(val_df)} images ({len(val_df)/len(df)*100:.1f}%)")
print(f"Test  : {len(test_df)} images ({len(test_df)/len(df)*100:.1f}%)")

Train : 714 images (70.0%)
Val   : 153 images (15.0%)
Test  : 153 images (15.0%)


## 9. Save Splits
Save the final splits as CSV files for use in model training.

In [9]:
train_df.to_csv(DATA_DIR / 'train.csv', index=False)
val_df.to_csv(DATA_DIR / 'val.csv', index=False)
test_df.to_csv(DATA_DIR / 'test.csv', index=False)

print(f"Saved: {DATA_DIR / 'train.csv'}")
print(f"Saved: {DATA_DIR / 'val.csv'}")
print(f"Saved: {DATA_DIR / 'test.csv'}")


✅ Saved: train.csv
✅ Saved: val.csv
✅ Saved: test.csv
